# Student Performance Regression Comparison

## 1. Select `.venv` Kernel

In VS Code, select the project `.venv` Python kernel before running the code cells.

## 2. Import Required Libraries

Import the numerical, plotting, and scikit-learn tools used below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


## 3. Generate Synthetic Dataset

Create 5,000 reproducible student records. The target score is generated from all listed predictors plus noise.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
row_count = 5_000

regression_data = pd.DataFrame(
    {
        "experience_years": rng.uniform(0, 20, row_count),
        "study_hours_per_week": rng.uniform(0, 40, row_count),
        "attendance_pct": rng.uniform(50, 100, row_count),
        "previous_exam_score": rng.uniform(30, 100, row_count),
        "assignment_completion_pct": rng.uniform(40, 100, row_count),
        "sleep_hours_per_night": rng.uniform(4, 10, row_count),
        "tutoring_support": rng.integers(0, 2, row_count),
    }
)

score_noise = rng.normal(0, 5, row_count)
regression_data["final_score"] = (
    35
    + 0.8 * regression_data["experience_years"]
    + 0.25 * regression_data["study_hours_per_week"]
    + 0.12 * regression_data["attendance_pct"]
    + 0.18 * regression_data["previous_exam_score"]
    + 0.1 * regression_data["assignment_completion_pct"]
    + 0.45 * regression_data["sleep_hours_per_night"]
    + 2.5 * regression_data["tutoring_support"]
    + score_noise
).clip(0, 100)

regression_data.head()

## 4. Save Dataset to CSV

Write the generated 5,000-row dataset to `regression_data.csv` in the project root.

In [ ]:
csv_path = "regression_data.csv"
regression_data.to_csv(csv_path, index=False)
print(f"Saved {len(regression_data):,} rows to {csv_path}")

## 5. Load Dataset and Split Data

Load the exported CSV, then create one reproducible 80/20 split shared by both models.

In [ ]:
data = pd.read_csv(csv_path)
target_column = "final_score"
feature_columns = [column for column in data.columns if column != target_column]

X = data[feature_columns]
y = data[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f"Dataset shape: {data.shape}")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Predictor features: {feature_columns}")

## 6. Simple Linear Regression (SLR)

Fit a one-feature model using only `experience_years`.

In [ ]:
simple_feature = ["experience_years"]
simple_model = LinearRegression()
simple_model.fit(X_train[simple_feature], y_train)
simple_predictions = simple_model.predict(X_test[simple_feature])
print(f"SLR coefficient for experience_years: {simple_model.coef_[0]:.3f}")

## 7. Multiple Linear Regression (MLR)

Fit the multiple-feature model using every predictor column in the dataset.

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)
multiple_predictions = multiple_model.predict(X_test)
print(f"MLR predictors: {len(feature_columns)}")

## 8. Evaluate Models (MAE, MSE, RMSE, $R^2$)

Compare both models using the same held-out test observations.

In [ ]:
model_predictions = {
    "SLR (experience_years)": simple_predictions,
    "MLR (all features)": multiple_predictions,
}
metric_rows = []

for model_name, predictions in model_predictions.items():
    mse = mean_squared_error(y_test, predictions)
    metric_rows.append(
        {
            "Model": model_name,
            "MAE": mean_absolute_error(y_test, predictions),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R^2": r2_score(y_test, predictions),
        }
    )

metrics = pd.DataFrame(metric_rows).set_index("Model")
metrics.round(3)

## 9. Plot Test Residuals

Plot actual minus predicted for the first 10 held-out observations for each model.

In [ ]:
sns.set_theme(style="whitegrid")
observations = np.arange(1, 11)
actual_values = y_test.iloc[:10].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for axis, (model_name, predictions) in zip(axes, model_predictions.items()):
    residuals = actual_values - predictions[:10]
    sns.barplot(x=observations, y=residuals, color="teal", ax=axis)
    axis.axhline(0, color="black", linewidth=1)
    axis.set_title(f"{model_name}: test residuals")
    axis.set_xlabel("Test sample")
    axis.set_ylabel("Residual (actual - predicted)")
    axis.set_xticks(observations - 1, labels=observations)

plt.tight_layout()
plt.show()

## 10. Check for Remaining Tasks

Run assertions for the requested dataset size, saved file, model features, metrics, and 10 plotted residuals per model.

In [ ]:
assert len(data) == 5_000
assert "experience_years" in feature_columns
assert len(simple_feature) == 1 and simple_feature[0] == "experience_years"
assert len(feature_columns) == 7
assert set(metrics.columns) == {"MAE", "MSE", "RMSE", "R^2"}
assert len(actual_values) == 10
assert all(len(predictions[:10]) == 10 for predictions in model_predictions.values())
assert np.isfinite(metrics.to_numpy()).all()

print("All notebook requirements are satisfied.")
print(f"Rows: {len(data):,}; random seed: {RANDOM_SEED}")
print(f"Saved CSV: {csv_path}")
print("Metrics: MAE, MSE, RMSE, R^2")
print("Residuals plotted: 10 per model")